# Image Classification with a CNN (64 classes)

A convolutional neural network built from scratch in Keras and trained in stages on a 64-class image dataset.

| | |
|---|---|
| **Task** | Multi-class image classification (64 classes, chance level ≈ 1.6%) |
| **Data** | 9,200 train / 2,301 val / 2,876 test images (fold 1 of a CSV split). Project dataset, **not included in this repo**. *TODO: add one line describing what the 64 classes are.* |
| **Model** | 3 conv blocks (64→128→256 filters) + Dense 128→64 with Dropout 0.3, 22.5M parameters, 224×224×3 input |
| **Result** | **83.1% test accuracy** (83.5% on the validation set at the end of training) |

## Approach
- Data augmentation on the training set: rotation 20°, width/height shift 0.2, shear 0.2, zoom 0.2, horizontal flip.
- Adam optimizer, categorical cross-entropy.
- Trained in four stages (30 epochs in total). The learning rate was raised from 1e-4 to 1e-3 after the first stage because training was too slow.

## Results

| Stage | Learning rate | Epochs | Test accuracy |
|---|---|---:|---:|
| 1 | 1e-4 | 10 | 46.5% |
| 2 | 1e-3 | 5 | 65.0% |
| 3 | 1e-3 | 10 | 79.3% |
| 4 | 1e-3 | 5 | **83.1%** |

Final epoch: train accuracy 77.3%, validation accuracy 83.5%. Training accuracy is lower than validation because augmentation and dropout are active during training. The curve was still improving when training stopped, so the model is probably under-trained.

## Limitations & next steps
- The test set was evaluated after every stage and training continued afterwards, so test results influenced when to stop. The validation accuracy (83.5%) is the cleaner number to quote; ideally decisions would be made on validation only.
- No pretrained baseline. A transfer-learning model (e.g. MobileNet or EfficientNet) would likely be faster to train and more accurate.
- Add a per-class report and confusion matrix to see which of the 64 classes are confused.

## How to run
The data-loading cell reads from a `DATA_DIR` variable. Set it to the folder that contains your images and the `csv/splits_csv/fold_1/{train,val,test}.csv` files (columns `image_path` and `label`). The Google Drive mount cell is only needed on Colab.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Importing Libs

In [2]:
import tensorflow
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, Dropout,Conv3D
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
import os


## Utilizing GPU

## Dataset Class

In [3]:
num_classes = 64

In [4]:
model = keras.models.Sequential()
model.add(keras.layers.Conv2D(64, (3, 3), activation='relu', input_shape=(224, 224, 3)))
model.add(keras.layers.MaxPooling2D((2, 2)))

model.add(keras.layers.Conv2D(128, (3, 3), activation='relu'))
model.add(keras.layers.MaxPooling2D((2, 2)))

model.add(keras.layers.Conv2D(256, (3, 3), activation='relu'))
model.add(keras.layers.MaxPooling2D((2, 2)))


model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(128, activation='relu'))
model.add(keras.layers.Dropout(0.3))
model.add(keras.layers.Dense(64, activation='relu'))
model.add(keras.layers.Dropout(0.3))
model.add(keras.layers.Dense(num_classes, activation='softmax'))

model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 173056)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    22,151,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,534,528 (85.96 MB)

 Trainable params: 22,534,528 (85.96 MB)

 Non-trainable params: 0 (0.00 B)

## Transforms

In [5]:
# Define image transformations / generators for Keras
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

In [6]:
# Set this to the folder that holds your images and the csv/ splits.
DATA_DIR = '/content/drive/Shareddrives/STAI_Project/datasets/'

import pandas as pd

# Load the CSV data splits
train_df = pd.read_csv(DATA_DIR + 'csv/splits_csv/fold_1/train.csv')
val_df = pd.read_csv(DATA_DIR + 'csv/splits_csv/fold_1/val.csv')
test_df = pd.read_csv(DATA_DIR + 'csv/splits_csv/fold_1/test.csv')

# Define directory containing your images (adjust this path if your images are located elsewhere)
image_directory = DATA_DIR

# Using flow_from_dataframe to handle the CSV mapping
train_dataset = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=image_directory,
    x_col='image_path',  # Replace with actual column name in your CSV for image paths
    y_col='label',       # Replace with actual column name in your CSV for labels
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

val_dataset = val_test_datagen.flow_from_dataframe(
    dataframe=val_df,
    directory=image_directory,
    x_col='image_path',
    y_col='label',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

test_dataset = val_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=image_directory,
    x_col='image_path',
    y_col='label',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

Found 9200 validated image filenames belonging to 64 classes.
Found 2301 validated image filenames belonging to 64 classes.
Found 2876 validated image filenames belonging to 64 classes.


## Evalutaion Function

In [7]:
model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])


## Training Function

In [8]:
model.fit(train_dataset, epochs=10, validation_data=val_dataset)


Epoch 1/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 5160s 18s/step - accuracy: 0.0253 - loss: 4.1196 - val_accuracy: 0.0665 - val_loss: 3.9204
Epoch 2/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 343s 1s/step - accuracy: 0.0626 - loss: 3.9060 - val_accuracy: 0.1682 - val_loss: 3.5436
Epoch 3/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 343s 1s/step - accuracy: 0.0992 - loss: 3.7004 - val_accuracy: 0.1812 - val_loss: 3.3204
Epoch 4/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 337s 1s/step - accuracy: 0.1290 - loss: 3.4985 - val_accuracy: 0.2151 - val_loss: 3.1305
Epoch 5/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 334s 1s/step - accuracy: 0.1686 - loss: 3.2996 - val_accuracy: 0.2625 - val_loss: 2.9316
Epoch 6/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 325s 1s/step - accuracy: 0.1890 - loss: 3.1394 - val_accuracy: 0.3425 - val_loss: 2.6127
Epoch 7/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 329s 1s/step - accuracy: 0.2188 - loss: 2.9908 - val_accuracy: 0.3807 - val_loss: 2.4190
Epoch 8/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 323s 1s/step - accuracy: 0.2422 - loss: 2.8457 - val_ac

## Calculate Metrics

In [9]:
model.evaluate(test_dataset)


90/90 ━━━━━━━━━━━━━━━━━━━━ 1481s 16s/step - accuracy: 0.4645 - loss: 2.0970


[2.096994400024414, 0.46453407406806946]

In [10]:
# Re-compile the model with a larger learning rate
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Continue training
history_continued = model.fit(
    train_dataset,
    epochs=5,
    validation_data=val_dataset
)

Epoch 1/5
288/288 ━━━━━━━━━━━━━━━━━━━━ 351s 1s/step - accuracy: 0.2183 - loss: 2.9512 - val_accuracy: 0.3803 - val_loss: 2.4716
Epoch 2/5
288/288 ━━━━━━━━━━━━━━━━━━━━ 326s 1s/step - accuracy: 0.3117 - loss: 2.4463 - val_accuracy: 0.4715 - val_loss: 1.8806
Epoch 3/5
288/288 ━━━━━━━━━━━━━━━━━━━━ 327s 1s/step - accuracy: 0.3721 - loss: 2.1743 - val_accuracy: 0.4946 - val_loss: 1.8050
Epoch 4/5
288/288 ━━━━━━━━━━━━━━━━━━━━ 328s 1s/step - accuracy: 0.4116 - loss: 2.0080 - val_accuracy: 0.6423 - val_loss: 1.2378
Epoch 5/5
288/288 ━━━━━━━━━━━━━━━━━━━━ 370s 1s/step - accuracy: 0.4638 - loss: 1.7880 - val_accuracy: 0.6510 - val_loss: 1.2418


In [11]:
model.evaluate(test_dataset)

90/90 ━━━━━━━━━━━━━━━━━━━━ 55s 606ms/step - accuracy: 0.6495 - loss: 1.2151


[1.2151248455047607, 0.6495131850242615]

In [12]:
history_continued = model.fit(
    train_dataset,
    epochs=10,
    validation_data=val_dataset
)

Epoch 1/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 330s 1s/step - accuracy: 0.4987 - loss: 1.7020 - val_accuracy: 0.6767 - val_loss: 1.1124
Epoch 2/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 331s 1s/step - accuracy: 0.5550 - loss: 1.5107 - val_accuracy: 0.6406 - val_loss: 1.3487
Epoch 3/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 323s 1s/step - accuracy: 0.5680 - loss: 1.4442 - val_accuracy: 0.6832 - val_loss: 1.0169
Epoch 4/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 322s 1s/step - accuracy: 0.5972 - loss: 1.3322 - val_accuracy: 0.8314 - val_loss: 0.5711
Epoch 5/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 382s 1s/step - accuracy: 0.6255 - loss: 1.2405 - val_accuracy: 0.7471 - val_loss: 0.8690
Epoch 6/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 332s 1s/step - accuracy: 0.6473 - loss: 1.1801 - val_accuracy: 0.7844 - val_loss: 0.7156
Epoch 7/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 323s 1s/step - accuracy: 0.6629 - loss: 1.1319 - val_accuracy: 0.8123 - val_loss: 0.6622
Epoch 8/10
288/288 ━━━━━━━━━━━━━━━━━━━━ 321s 1s/step - accuracy: 0.6908 - loss: 1.0341 - val_accu

In [13]:
model.evaluate(test_dataset)

90/90 ━━━━━━━━━━━━━━━━━━━━ 57s 636ms/step - accuracy: 0.7935 - loss: 0.7874


[0.7873677015304565, 0.7934631705284119]

In [14]:
history_continued = model.fit(
    train_dataset,
    epochs=5,
    validation_data=val_dataset
)

Epoch 1/5
288/288 ━━━━━━━━━━━━━━━━━━━━ 331s 1s/step - accuracy: 0.7240 - loss: 0.9031 - val_accuracy: 0.8444 - val_loss: 0.5708
Epoch 2/5
288/288 ━━━━━━━━━━━━━━━━━━━━ 333s 1s/step - accuracy: 0.7442 - loss: 0.8277 - val_accuracy: 0.8640 - val_loss: 0.5322
Epoch 3/5
288/288 ━━━━━━━━━━━━━━━━━━━━ 330s 1s/step - accuracy: 0.7496 - loss: 0.8265 - val_accuracy: 0.8514 - val_loss: 0.5535
Epoch 4/5
288/288 ━━━━━━━━━━━━━━━━━━━━ 308s 1s/step - accuracy: 0.7688 - loss: 0.7798 - val_accuracy: 0.8774 - val_loss: 0.4576
Epoch 5/5
288/288 ━━━━━━━━━━━━━━━━━━━━ 315s 1s/step - accuracy: 0.7726 - loss: 0.7561 - val_accuracy: 0.8353 - val_loss: 0.6646


In [15]:
model.evaluate(test_dataset)

90/90 ━━━━━━━━━━━━━━━━━━━━ 54s 597ms/step - accuracy: 0.8307 - loss: 0.6084


[0.608401358127594, 0.8306676149368286]